# 07 -- MLP classifier + LSTM regressor

Данные уже подготовлены в [06_LSTM_Data_Preparation.ipynb](./06_LSTM_Data_Preparation.ipynb), а табличные признаки -- в [05_Data-Modeling.ipynb](./05_Data-Modeling.ipynb).

Делаю простую hurdle-модель:

1. MLP по static-признакам предсказывает вероятность покупки $p=P(GMV>0)$;
2. LSTM учу только на пользователях с ненулевым target и предсказываю положительный GMV;
3. перемножаю два прогноза: $\hat y_{soft}=p\cdot \hat y_{LSTM}$;
4. в самом конце зануляю слишком маленькие $\hat y_{soft}$.

Нейросети больше не гоняю по grid search. Беру `AdamW(lr=5e-4, weight_decay=5e-4)` -- он был лучшим в предыдущем запуске.

## Импорты и настройки

Для подбора финального порога использую обычный `KFold + cross_val_score`. Нейросети внутри CV не переобучаются, поэтому эта часть работает быстро.

In [7]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.metrics import make_scorer
from sklearn.model_selection import KFold, cross_val_score
from torch import nn
from torch.utils.data import ConcatDataset, DataLoader, Dataset


def find_project_root():
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "data" / "lstm" / "meta.json").exists():
            return path


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "lstm"
SUBMISSION_DIR = PROJECT_ROOT / "submissions"
SUBMISSION_DIR.mkdir(exist_ok=True)

with open(DATA_DIR / "meta.json", encoding="utf-8") as f:
    META = json.load(f)

LABELED_CUTOFFS = META["labeled_cutoffs"]
INFERENCE_CUTOFF = META["inference_cutoff"]
BASE_FEATURES = META["base_sequence_features"]
CALENDAR_FEATURES = META["calendar_sequence_features"]
STATIC_FEATURES = META["static_features"]
STATIC_LOG_FEATURES = META["static_log_copy_features"]

BASE_INDEX = {name: i for i, name in enumerate(BASE_FEATURES)}
STATIC_INDEX = {name: i for i, name in enumerate(STATIC_FEATURES)}
STATIC_LOG_INDICES = [STATIC_INDEX[name] for name in STATIC_LOG_FEATURES]

TRAIN_CUTOFFS = LABELED_CUTOFFS[:8]
VAL_CUTOFF = LABELED_CUTOFFS[8]
HOLDOUT_CUTOFF = LABELED_CUTOFFS[9]

BATCH_SIZE = 1024
CLASSIFIER_EPOCHS = 6
LSTM_EPOCHS = 13
LR = 5e-4  # лучшие параметры из одноименного ноутбука в предыдущем коммите
WEIGHT_DECAY = 5e-4
RANDOM_STATE = 42

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print("device:", DEVICE)

device: mps


## Данные

Для classifier беру 95 static-признаков из [Prepared_data.parquet](../../data/Prepared_data.parquet): 91 признак из [05_Data-Modeling.ipynb](./05_Data-Modeling.ipynb) + 4 календарных.

К ним на лету добавляю `log1p`-копии heavy-tail признаков и маски пропусков.

Для LSTM беру последние 90 дней. В [06_LSTM_Data_Preparation.ipynb](./06_LSTM_Data_Preparation.ipynb) сохранены 13 базовых и 6 календарных каналов. Еще 21 временной признак считаю на батче: conversion rate, rolling mean и разности.

In [8]:
class StaticDataset(Dataset):
    def __init__(self, cutoff, with_target=True):
        path = DATA_DIR / cutoff
        self.X = np.load(path / "static.npy", mmap_mode="r")
        self.users = np.load(path / "user_id.npy", mmap_mode="r")
        self.y = np.load(path / "y.npy", mmap_mode="r") if with_target else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        x = torch.from_numpy(np.array(self.X[i], dtype=np.float32))
        if self.y is None:
            return x
        return x, torch.tensor(float(self.y[i]), dtype=torch.float32)


class SequenceDataset(Dataset):
    def __init__(self, cutoff, with_target=True, positive_only=False):
        path = DATA_DIR / cutoff
        self.X = np.load(path / "X.npy", mmap_mode="r")
        self.calendar = np.load(path / "calendar.npy", mmap_mode="r").astype(np.float32)
        self.users = np.load(path / "user_id.npy", mmap_mode="r")
        self.y = np.load(path / "y.npy", mmap_mode="r") if with_target else None
        self.indices = np.flatnonzero(self.y > 0) if positive_only else None

    def __len__(self):
        return len(self.X) if self.indices is None else len(self.indices)

    def __getitem__(self, i):
        j = i if self.indices is None else int(self.indices[i])

        x = np.concatenate([
            np.array(self.X[j], dtype=np.float32),
            self.calendar,
        ], axis=1)
        x = torch.from_numpy(x)

        if self.y is None:
            return x
        return x, torch.tensor(float(self.y[j]), dtype=torch.float32)


def make_loader(dataset_class, cutoffs, shuffle=False, **kwargs):
    if isinstance(cutoffs, str):
        cutoffs = [cutoffs]

    dataset = ConcatDataset([
        dataset_class(cutoff, **kwargs)
        for cutoff in cutoffs
    ])

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        drop_last=shuffle,
    )

## Признаки classifier

Оставляю те же static-признаки, которые уже подготовил раньше. Для тяжелых положительных величин добавляю `log1p`-копии, а для пропусков -- отдельные бинарные маски.

In [9]:
STATIC_LOG_INDICES = torch.tensor(STATIC_LOG_INDICES, dtype=torch.long)
STATIC_INPUT_SIZE = len(STATIC_FEATURES) + len(STATIC_LOG_FEATURES) + len(STATIC_FEATURES)


def make_static_features(x):
    log_source = x.index_select(1, STATIC_LOG_INDICES.to(x.device))
    logs = torch.log1p(torch.nan_to_num(log_source, nan=0.0).clamp_min(0))
    missing = (~torch.isfinite(x)).float()
    x = torch.nan_to_num(x)
    return torch.cat([x, logs, missing], dim=1)


print("classifier input:", STATIC_INPUT_SIZE)

classifier input: 241


## 1. MLP classifier

Classifier больше не прогоняю через LSTM. Его задача проще: по агрегированному поведению пользователя оценить вероятность того, что GMV следующих 30 дней будет больше нуля.

In [10]:
class PurchaseMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(STATIC_INPUT_SIZE),
            nn.Linear(STATIC_INPUT_SIZE, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(make_static_features(x)).squeeze(1)


def fit_classifier(cutoffs, epochs=CLASSIFIER_EPOCHS):
    loader = make_loader(StaticDataset, cutoffs, shuffle=True)
    model = PurchaseMLP().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCELoss()

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for x, y in loader:
            x = x.to(DEVICE)
            y = (y.to(DEVICE) > 0).float()

            optimizer.zero_grad()
            pred = model(x)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(y)

        print(f"classifier | epoch {epoch:02d}/{epochs:02d} | loss={total_loss / len(loader.dataset):.5f}")

    return model


classifier = fit_classifier(TRAIN_CUTOFFS)

classifier | epoch 01/06 | loss=0.47927
classifier | epoch 02/06 | loss=0.47689
classifier | epoch 03/06 | loss=0.47622
classifier | epoch 04/06 | loss=0.47605
classifier | epoch 05/06 | loss=0.47585
classifier | epoch 06/06 | loss=0.47568


## Признаки LSTM

Сохраняю сильную часть прошлой версии: rolling mean, conversion rate и разности. Они считаются прямо на батче и не занимают место на диске.

In [11]:
def causal_mean(x, window):
    x = F.pad(x.unsqueeze(1), (window - 1, 0))
    return F.avg_pool1d(x, window, stride=1).squeeze(1)


def first_difference(x):
    return F.pad(x[:, 1:] - x[:, :-1], (1, 0))


def safe_ratio(a, b, max_value=5.0):
    ratio = a / b.clamp_min(1e-3)
    return torch.where(b > 0, ratio, torch.zeros_like(ratio)).clamp(0, max_value)


def make_sequence_features(x):
    searches_log = x[..., BASE_INDEX["searches"]]
    search_to_cart_log = x[..., BASE_INDEX["search_to_cart"]]
    search_to_ord_log = x[..., BASE_INDEX["search_to_ord"]]
    cat_to_cart_log = x[..., BASE_INDEX["cat_to_cart"]]
    cat_to_ord_log = x[..., BASE_INDEX["cat_to_ord"]]
    to_cart_log = x[..., BASE_INDEX["to_cart"]]
    to_ord_log = x[..., BASE_INDEX["to_ord"]]
    gmv_search_log = x[..., BASE_INDEX["gmv_search"]]
    gmv_log = x[..., BASE_INDEX["gmv"]]
    active = x[..., BASE_INDEX["active"]]

    searches = torch.expm1(searches_log)
    search_to_cart = torch.expm1(search_to_cart_log)
    search_to_ord = torch.expm1(search_to_ord_log)
    cat_to_cart = torch.expm1(cat_to_cart_log)
    cat_to_ord = torch.expm1(cat_to_ord_log)
    to_cart = torch.expm1(to_cart_log)
    to_ord = torch.expm1(to_ord_log)
    gmv_search = torch.expm1(gmv_search_log)
    gmv = torch.expm1(gmv_log)

    derived = [
        (search_to_cart > 0).float(),
        (search_to_ord > 0).float(),
        (cat_to_cart > 0).float(),
        (cat_to_ord > 0).float(),
        safe_ratio(search_to_cart, searches),
        safe_ratio(search_to_ord, searches),
        safe_ratio(to_ord, to_cart),
        safe_ratio(gmv_search, gmv, 1.5),
    ]

    for values in [searches_log, to_cart_log, to_ord_log, gmv_log]:
        derived += [causal_mean(values, 7), causal_mean(values, 30)]

    derived += [
        causal_mean(active, 7),
        causal_mean(active, 30),
        first_difference(searches_log),
        first_difference(to_ord_log),
        first_difference(gmv_log),
    ]

    return torch.cat([x, torch.stack(derived, dim=-1)], dim=-1)


SEQ_INPUT_SIZE = len(BASE_FEATURES) + len(CALENDAR_FEATURES) + 21
print("lstm input:", SEQ_INPUT_SIZE)

lstm input: 40


## 2. LSTM regressor

Regressor учу только на `target > 0`. Он предсказывает `log1p(GMV)`, потому что итоговая метрика -- RMSLE.

Архитектуру оставляю простой: 2-layer BiLSTM, dropout и небольшая полносвязная голова.

In [12]:
class GMVLSTM(nn.Module):
    def __init__(self, hidden_size=128):
        super().__init__()
        self.norm = nn.LayerNorm(SEQ_INPUT_SIZE)
        self.lstm = nn.LSTM(
            input_size=SEQ_INPUT_SIZE,
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True,
            dropout=0.25,
            bidirectional=True,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.norm(make_sequence_features(x))
        _, (hidden, _) = self.lstm(x)
        hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.head(hidden).squeeze(1)


def fit_lstm(cutoffs, epochs=LSTM_EPOCHS):
    loader = make_loader(SequenceDataset, cutoffs, shuffle=True, positive_only=True)
    model = GMVLSTM().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.MSELoss()

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for x, y in loader:
            x = x.to(DEVICE)
            y = torch.log1p(y.to(DEVICE))

            optimizer.zero_grad()
            pred = model(x)
            loss = loss_fn(pred, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item() * len(y)

        print(f"lstm       | epoch {epoch:02d}/{epochs:02d} | loss={total_loss / len(loader.dataset):.5f}")

    return model


lstm = fit_lstm(TRAIN_CUTOFFS)

lstm       | epoch 01/13 | loss=1.67646
lstm       | epoch 02/13 | loss=1.37860
lstm       | epoch 03/13 | loss=1.36643
lstm       | epoch 04/13 | loss=1.35599
lstm       | epoch 05/13 | loss=1.35036
lstm       | epoch 06/13 | loss=1.34324
lstm       | epoch 07/13 | loss=1.33821
lstm       | epoch 08/13 | loss=1.33188
lstm       | epoch 09/13 | loss=1.32787
lstm       | epoch 10/13 | loss=1.32187
lstm       | epoch 11/13 | loss=1.31827
lstm       | epoch 12/13 | loss=1.31558
lstm       | epoch 13/13 | loss=1.31101


## Предсказания моделей

Classifier возвращает мягкую вероятность покупки, LSTM -- положительный GMV. Никаких жестких классов здесь нет.

In [13]:
@torch.no_grad()
def predict_classifier(model, cutoff, with_target=True):
    dataset = StaticDataset(cutoff, with_target)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE)
    model.eval()

    pred, target = [], []

    for batch in loader:
        if with_target:
            x, y = batch
            target.append(y.numpy())
        else:
            x = batch

        pred.append(model(x.to(DEVICE)).cpu().numpy())

    y = np.concatenate(target) if with_target else None
    return np.asarray(dataset.users), np.concatenate(pred), y


@torch.no_grad()
def predict_lstm(model, cutoff, with_target=True):
    dataset = SequenceDataset(cutoff, with_target)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE)
    model.eval()

    pred, target = [], []

    for batch in loader:
        if with_target:
            x, y = batch
            target.append(y.numpy())
        else:
            x = batch

        pred_log = model(x.to(DEVICE)).cpu().numpy()
        pred.append(np.expm1(np.clip(pred_log, 0, None)))

    y = np.concatenate(target) if with_target else None
    return np.asarray(dataset.users), np.concatenate(pred), y


def predict_soft(classifier, lstm, cutoff, with_target=True):
    users, probability, y = predict_classifier(classifier, cutoff, with_target)
    _, gmv, _ = predict_lstm(lstm, cutoff, with_target)
    return users, probability * gmv, y

## 3. Финальный threshold

Threshold применяю только после `probability * gmv_prediction`.

Это порог **финального прогноза GMV в рублях**, а не threshold по реальным покупкам и не RMSLE. Поэтому значения `1`, `5`, `20` и т.д. здесь имеют нормальный смысл.

Проверяю 8 значений. `cross_val_score` только делит готовые validation predictions на 5 частей -- LSTM и MLP при этом не переобучаются.

In [14]:
def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))


class ThresholdModel(BaseEstimator, RegressorMixin):
    def __init__(self, threshold=0.0):
        self.threshold = threshold

    def fit(self, X, y=None):
        return self

    def predict(self, X):
        pred = np.asarray(X).ravel()
        return np.where(pred >= self.threshold, pred, 0.0)


_, val_soft, y_val = predict_soft(classifier, lstm, VAL_CUTOFF)

thresholds = [0, 1, 2, 5, 10, 20, 50, 100]
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scorer = make_scorer(rmsle, greater_is_better=False)

rows = []
for threshold in thresholds:
    score = -cross_val_score(
        ThresholdModel(threshold),
        val_soft.reshape(-1, 1),
        y_val,
        scoring=scorer,
        cv=cv,
    ).mean()
    rows.append((threshold, score))

threshold_results = pd.DataFrame(rows, columns=["threshold", "rmsle"]).sort_values("rmsle")
best_threshold = float(threshold_results.iloc[0]["threshold"])

threshold_results

,threshold,rmsle
5,20,1.976057
4,10,1.990201
3,5,2.059151
2,2,2.081308
1,1,2.081523
0,0,2.081530
6,50,2.292877
7,100,2.718289


## Holdout

После выбора threshold один раз проверяю всю схему на последнем размеченном cutoff.

In [15]:
_, holdout_soft, y_holdout = predict_soft(classifier, lstm, HOLDOUT_CUTOFF)
holdout_pred = np.where(holdout_soft >= best_threshold, holdout_soft, 0.0)

print("best threshold:", best_threshold)
print("holdout RMSLE:", f"{rmsle(y_holdout, holdout_pred):.6f}")
print("zero predictions:", f"{(holdout_pred == 0).mean():.2%}")

best threshold: 20.0
holdout RMSLE: 1.943656
zero predictions: 39.80%


## 4. Финальное обучение

После проверки просто переобучаю classifier и LSTM на всех размеченных cutoff. Архитектуру, optimizer и число эпох больше не подбираю.

In [16]:
final_classifier = fit_classifier(LABELED_CUTOFFS)
final_lstm = fit_lstm(LABELED_CUTOFFS)

classifier | epoch 01/06 | loss=0.47812
classifier | epoch 02/06 | loss=0.47593
classifier | epoch 03/06 | loss=0.47548
classifier | epoch 04/06 | loss=0.47520
classifier | epoch 05/06 | loss=0.47486
classifier | epoch 06/06 | loss=0.47467
lstm       | epoch 01/13 | loss=1.64118
lstm       | epoch 02/13 | loss=1.38269
lstm       | epoch 03/13 | loss=1.36827
lstm       | epoch 04/13 | loss=1.35807
lstm       | epoch 05/13 | loss=1.34747
lstm       | epoch 06/13 | loss=1.33957
lstm       | epoch 07/13 | loss=1.33133
lstm       | epoch 08/13 | loss=1.32524
lstm       | epoch 09/13 | loss=1.32045
lstm       | epoch 10/13 | loss=1.31487
lstm       | epoch 11/13 | loss=1.31025
lstm       | epoch 12/13 | loss=1.30612
lstm       | epoch 13/13 | loss=1.30237


## Submission

Финальный файл сохраняю в [submissions/lstm.csv](../../submissions/lstm.csv).

In [17]:
user_ids, soft_pred, _ = predict_soft(
    final_classifier,
    final_lstm,
    INFERENCE_CUTOFF,
    with_target=False,
)

pred = np.where(soft_pred >= best_threshold, soft_pred, 0.0)

sample = pd.read_csv(PROJECT_ROOT / "data" / "sample_submit.csv")
pred_by_user = pd.Series(pred, index=user_ids)

submission = sample.copy()
submission["predict"] = submission["user_id"].map(pred_by_user)
submission.to_csv(SUBMISSION_DIR / "lstm.csv", index=False)

print("zero predictions:", f"{(submission['predict'] == 0).mean():.2%}")
submission.head()

zero predictions: 46.76%


,user_id,predict
0,2,0.00000
1,7,184.33786
2,15,40.82690
3,18,165.20549
4,23,0.00000
